In [50]:
import pdfplumber
from typing import List, Tuple, Any, Iterable
import re


In [2]:
def bbox_intersects(a, b):
    # a and b = [x0, top, x1, bottom]
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    horiz = not (ax1 <= bx0 or bx1 <= ax0)
    vert = not (ay1 <= by0 or by1 <= ay0)
    return horiz and vert


In [3]:
def get_page_elements(pdf_path):
    pages_output = []
    with pdfplumber.open(pdf_path) as pdf:
        for pidx, page in enumerate(pdf.pages, start=1):
            # 1) find table bboxes (pdfplumber Table objects via find_tables)
            table_bboxes = []
            tables = []
            try:
                table_objs = page.find_tables()
            except Exception:
                table_objs = []
            for t_idx, t in enumerate(table_objs):
                # pdfplumber Table has .bbox and .extract() method
                tbbox = t.bbox  # (x0, top, x1, bottom)
                table_bboxes.append({"bbox": tbbox, "table_index": t_idx})
                # get rows as list of lists
                try:
                    rows = t.extract()
                except Exception:
                    # fallback to page.extract_tables
                    rows = None
                tables.append({"table_index": t_idx, "bbox": tbbox, "rows": rows})

            # 2) extract words with bboxes and filter out words that fall into any table bbox
            words = page.extract_words()  # each item has x0, x1, top (y0), bottom (y1), text
            narrative_words = []
            for w in words:
                wbox = (float(w["x0"]), float(w["top"]), float(w["x1"]), float(w["bottom"]))
                in_table = any(bbox_intersects(wbox, tb["bbox"]) for tb in table_bboxes)
                if not in_table:
                    narrative_words.append(w)
            # 3) rebuild narrative text using word order grouping by line (simple)
            # group by 'top' with small tolerance to form lines
            lines = {}
            tol = 3  # pixels tolerance
            for w in narrative_words:
                top = round(float(w["top"]) / tol) * tol
                lines.setdefault(top, []).append((float(w["x0"]), w["text"]))
            # sort lines by top, then words by x0
            sorted_lines = [ " ".join([t[1] for t in sorted(line, key=lambda x: x[0])]) for top, line in sorted(lines.items()) ]

            narrative_text = "\n".join(sorted_lines).strip()
            # 4) for the page output, include tables (as DataFrames if available)
            table_records = []
            for t in tables:
#                 if t["rows"]:
# #                     df = pd.DataFrame(t["rows"][1:], columns=t["rows"][0]) if len(t["rows"])>1 else pd.DataFrame(t["rows"])
# #                     df = df.to_dict(orient='records')
#                     headers = t["rows"][0]
#                     data_rows = t["rows"][1:]
#                     # Use a list comprehension to build the list of dictionaries
#                     records = [dict(zip(headers, row)) for row in data_rows]

#                 else:
#                     # fallback: try page.extract_tables and pick the matching by bbox proximity
#                     records = None
                table_records.append({"table_index": t["table_index"], "bbox": t["bbox"], "table_dct": t['rows']})

            pages_output.append({
                "page": pidx,
                "narrative_text": narrative_text,
                "tables": table_records
            })
    return pages_output

In [4]:
pdf_pages = get_page_elements("assets/CP_Perjalanan Dinas.pdf")
pdf_pages

[{'page': 1, 'narrative_text': 'PERJALANAN DINAS\nAUG 2025', 'tables': []},
 {'page': 2,
  'narrative_text': 'DEFINITION\nPERJALANAN DINAS adalah perjalanan yang dilakukan oleh Karyawan Perusahaan menuju lokasi di luar Lokasi Kerja Karyawan dan/atau kembali\nkeLokasiKerjaKaryawandalamrangkapelaksanaantugasataukepentinganPerusahaan.\nPERJALANAN DINAS JARAK DEKAT, disingkat sebagai PDJD, adalah Perjalanan Dinas dalam rangka pelaksanaan tugas atau kepentingan\nPerusahaan ke suatu wilayah dengan jarak antara Lokasi Kerja Karyawan dengan tujuan dinas <125 km (kurang dari seratus dua puluh lima\nkilometer)dalam1(satu)kaliperjalanan.\nPERJALANANDINASJARAKJAUH,disingkatsebagaiPDJJ,adalahPerjalananDinas dalamrangkapelaksanaantugasataukepentinganPerusahaan\nke suatu wilayah di dalam negeri maupun luar negeri negeri dengan jarak antara Lokasi Kerja Karyawan dengan tujuan dinas ≥125 km (lebih dari\natausamadenganseratusduapuluhlimakilometer)dalam1(satu)kaliperjalanandandapatmemilikiunsurmenginap.\

In [12]:
pdf_pages[-1]

{'page': 27,
 'narrative_text': 'BIAYA AKOMODASI UNTUK PERJALANAN DINAS2)\n1)SelainSingapore,Hongkong,danJepang..\n2)Hargasebelumpajak.',
 'tables': [{'table_index': 0,
   'bbox': (12.0, 60.0, 702.27, 347.762),
   'table_dct': [['Gol.',
     'Dalam Negeri (dalam ribuan Rupiah)',
     None,
     None,
     None,
     None,
     'Luar Negeri (dalam USD)',
     None],
    [None,
     'Sumatera',
     'Jawa & Bali',
     'Kalimantan\n(selain Kaltim)',
     'Kaltim &\nSulawesi',
     'Papua',
     'Asia1), Afrika\n(dalam USD)',
     'Non Asia& Afrika,\nSingapore, Hongkong dan\nJepang (dalam USD)'],
    ['10',
     'Sesuai tagihan, Hotel Bintang 5 (lima)',
     None,
     None,
     None,
     None,
     None,
     None],
    ['9', '1.200', '1.500', '1.200', '1.200', '1.500', '300', '400'],
    ['8', '1.000', '1.350', '1.000', '1.000', '1.350', '250', '350'],
    ['7', '900', '1200', '900', '900', '1.200', '200', '250'],
    ['6', '800', '900', '750', '800', '900', '80', '150'],
    ['5', '6

In [5]:
pdf_pages[6]['tables'][1]['table_dct']

[['Jenis Biaya', 'PDJD', 'PDJD Menginap', 'PDJJ', 'PDJJ Tidak\nMenginap'],
 ['Uang Saku Dinas',
  '-',
  'Sesuai ketentuan pemberian Uang Saku\nDinas',
  None,
  '-'],
 ['Uang Makan Dinas',
  '-',
  '-',
  'Sesuai ketentuan pemberian Uang\nMakan Dinas',
  None],
 ['Biaya Transportasi Dinas',
  'SesuaiketentuanpemberianBiayaTransportasiDinas',
  None,
  None,
  None],
 ['Biaya Akomodasi',
  '-',
  'Sesuai ketentuan pemberian Biaya\nAkomodasi Dinas',
  None,
  '-'],
 ['Biaya Lain-lain',
  'Diberikanjikaditetapkanoleh Perusahaan yang ditentukanoleh CEO (untukKaryawanBusiness\nUnit) atauDireksi(untukKaryawanCorporate) sebagaibagiandaribiayaPerjalananDinas.',
  None,
  None,
  None],
 ['Cash Advance', 'Tidak Diperbolehkan', None, 'Diperbolehkan', None]]

In [39]:
# !pip install fuzzywuzzy
# !pip install python-Levenshtein
from fuzzywuzzy import fuzz, process

In [7]:
def check_nested(current_cell, tables):
    __, score = process.extractOne(current_cell, tables)
    if score > 80:
        return True
    return False

In [9]:
tables = []
for page in pdf_pages:
    for table in page['tables']:
        tables.append(table['table_dct'])
tables

[[['', 'PERJALANAN DINAS/ BUSINESS TRAVEL'],
  ['Dilakukan oleh', 'KaryawanKontrakatau KaryawanTetap.'],
  ['PersetujuanDinas',
   '1. PDJDtanpaunsurmenginap:SPPDwajibdisetujuiAtasanLangsungKaryawandenganGolonganminimal6.\n2. PDJD menginap1) : SPPD wajib disetujui Atasan Karyawan minimal Department Head denganmelampirkan justifikasi\ntertulis.\n3. PDJJdalamnegeri:SPPDwajibdisetujuiAtasanKaryawanminimalDepartmentHead.\n4. PDJJ luar negeri: SPPD wajib disetujui hingga CEO untuk Karyawan Business Unit atau Direksi untuk Karyawan\nCorporate.\n1)PDJD dengan unsur menginap dapat dilakukan dengan pertimbangan situasi yang tidak dapat terhindarkan atau\ndipandanglebihefektifdanefisiendarisisibiayadantenaga.'],
  ['Tugasyang\ndikategorikan\nPerjalananDinas',
   'TugaskedinasanyangtermasukdalamkategoriPerjalananDinasantaralain:\n1. MelakukantugaskedinasankegrupusahaPerusahaan.\n2. Mewakili Perusahaan untuk memenuhi undangan dinas instansi pemerintah organisasi/lembaga swasta untuk\nforumsimposiu

In [10]:
tables[0]

[['', 'PERJALANAN DINAS/ BUSINESS TRAVEL'],
 ['Dilakukan oleh', 'KaryawanKontrakatau KaryawanTetap.'],
 ['PersetujuanDinas',
  '1. PDJDtanpaunsurmenginap:SPPDwajibdisetujuiAtasanLangsungKaryawandenganGolonganminimal6.\n2. PDJD menginap1) : SPPD wajib disetujui Atasan Karyawan minimal Department Head denganmelampirkan justifikasi\ntertulis.\n3. PDJJdalamnegeri:SPPDwajibdisetujuiAtasanKaryawanminimalDepartmentHead.\n4. PDJJ luar negeri: SPPD wajib disetujui hingga CEO untuk Karyawan Business Unit atau Direksi untuk Karyawan\nCorporate.\n1)PDJD dengan unsur menginap dapat dilakukan dengan pertimbangan situasi yang tidak dapat terhindarkan atau\ndipandanglebihefektifdanefisiendarisisibiayadantenaga.'],
 ['Tugasyang\ndikategorikan\nPerjalananDinas',
  'TugaskedinasanyangtermasukdalamkategoriPerjalananDinasantaralain:\n1. MelakukantugaskedinasankegrupusahaPerusahaan.\n2. Mewakili Perusahaan untuk memenuhi undangan dinas instansi pemerintah organisasi/lembaga swasta untuk\nforumsimposium/disk

In [13]:
def clean_table(table):
    '''
    clean None with previous cell (based on result, None value is the previous. Ex: CP_Perjalanan Dinas)
    '''
    for row in table:
        for i, cell in enumerate(row):
            if cell==None and i>0:
                row[i] = row[i-1]
    #             print('!!')

    return table

In [29]:
def get_all_tables(pdf_pages):
    tables = []
    tables_flatted = []
    for page in pdf_pages:
        for table in page['tables']:
            tables.append(table['table_dct'])
            txt = ''
            for row in table['table_dct']:
                for col in row:
                    col = '' if not col else col
                    txt += ' ' + col
                txt += '\n'
            tables_flatted.append(txt)
    return tables, tables_flatted

In [51]:
def get_all_tables(pdf_pages: Iterable[dict]) -> Tuple[List[Any], List[str]]:
    """
    Return (tables, tables_flatted)
    - tables: list of each table['table_dct'] (or None if missing)
    - tables_flatted: string for each table where cells in a row are joined by spaces
      and rows are joined by newlines.
    Behavior:
      - treats None cells as empty string (but preserves 0 and False as values).
      - tolerant if 'tables' or 'table_dct' keys are missing.
    """
    tables: List[Any] = []
    tables_flatted: List[str] = []

    append_table = tables.append
    append_flat = tables_flatted.append

    for page in pdf_pages:
        # safer access to page['tables']
        for table in page.get("tables", []):
            # prefer dict-style access but be robust to different shapes
            td = table.get("table_dct") if isinstance(table, dict) else getattr(table, "table_dct", None)
            append_table(td)

            # guard: if td is not an iterable of rows, use empty list
            if td is None:
                append_flat("")  # keep alignment: one flat string per table entry
                continue

            # build flattened text efficiently using join() (linear time)
            # convert None -> '' but keep falsy numerics (0) as '0'
            flat = "\n".join(
                " ".join(
                    ("" if cell is None else str(cell))
                    for cell in row
                )
                for row in td
            )
            flat = re.sub(r'\s+', ' ', flat)

            append_flat(flat)

    return tables, tables_flatted

In [52]:
tables, tables_flattened = get_all_tables(pdf_pages)
tables_flattened

[' PERJALANAN DINAS/ BUSINESS TRAVEL Dilakukan oleh KaryawanKontrakatau KaryawanTetap. PersetujuanDinas 1. PDJDtanpaunsurmenginap:SPPDwajibdisetujuiAtasanLangsungKaryawandenganGolonganminimal6. 2. PDJD menginap1) : SPPD wajib disetujui Atasan Karyawan minimal Department Head denganmelampirkan justifikasi tertulis. 3. PDJJdalamnegeri:SPPDwajibdisetujuiAtasanKaryawanminimalDepartmentHead. 4. PDJJ luar negeri: SPPD wajib disetujui hingga CEO untuk Karyawan Business Unit atau Direksi untuk Karyawan Corporate. 1)PDJD dengan unsur menginap dapat dilakukan dengan pertimbangan situasi yang tidak dapat terhindarkan atau dipandanglebihefektifdanefisiendarisisibiayadantenaga. Tugasyang dikategorikan PerjalananDinas TugaskedinasanyangtermasukdalamkategoriPerjalananDinasantaralain: 1. MelakukantugaskedinasankegrupusahaPerusahaan. 2. Mewakili Perusahaan untuk memenuhi undangan dinas instansi pemerintah organisasi/lembaga swasta untuk forumsimposium/diskusipanel,ataupengurusanlegalitasPerusahaan. 3. 

In [53]:
tables_flattened[3]

'Jenis Biaya PDJD PDJD Menginap PDJJ PDJJ Tidak Menginap Uang Saku Dinas - Sesuai ketentuan pemberian Uang Saku Dinas - Uang Makan Dinas - - Sesuai ketentuan pemberian Uang Makan Dinas Biaya Transportasi Dinas SesuaiketentuanpemberianBiayaTransportasiDinas Biaya Akomodasi - Sesuai ketentuan pemberian Biaya Akomodasi Dinas - Biaya Lain-lain Diberikanjikaditetapkanoleh Perusahaan yang ditentukanoleh CEO (untukKaryawanBusiness Unit) atauDireksi(untukKaryawanCorporate) sebagaibagiandaribiayaPerjalananDinas. Cash Advance Tidak Diperbolehkan Diperbolehkan '

In [68]:
pdf_pages[6]['tables'][0]['table_dct']

[['', 'PERJALANAN DINAS/ BUSINESS TRAVEL'],
 ['PenggantianBiaya\nPerjalananDinas',
  'PDJJ Tidak\nJenis Biaya PDJD PDJD Menginap PDJJ\nMenginap\nUang Saku Dinas - Sesuai ketentuan pemberian Uang Saku -\nDinas\nUang Makan Dinas - - Sesuai ketentuan pemberian Uang\nMakan Dinas\nBiaya Transportasi Dinas SesuaiketentuanpemberianBiayaTransportasiDinas\nBiaya Akomodasi - Sesuai ketentuan pemberian Biaya -\nAkomodasi Dinas\nBiaya Lain-lain Diberikanjikaditetapkanoleh Perusahaan yang ditentukanoleh CEO (untukKaryawanBusiness\nUnit) atauDireksi(untukKaryawanCorporate) sebagaibagiandaribiayaPerjalananDinas.\nCash Advance Tidak Diperbolehkan Diperbolehkan'],
 ['AlokasiBiayaDinas',
  'KetentuanPerjalananDinasbagiKaryawanyangmelakukanPerjalananDinasakanmengikutiketentuanyangberlakudi\nBusinessUnit-nyamasing-masingbagiKaryawanBusinessUnit danketentuanyangberlakudiCorporate bagiKaryawan\nCorporate, tetapi alokasi biaya-biaya yang timbul akan dibebankan sesuai dengan kepentingan pelaksanaan Perjalanan

In [58]:
table_6 = pdf_pages[6]['tables'][0]['table_dct'][1][1]
table_6

'PDJJ Tidak\nJenis Biaya PDJD PDJD Menginap PDJJ\nMenginap\nUang Saku Dinas - Sesuai ketentuan pemberian Uang Saku -\nDinas\nUang Makan Dinas - - Sesuai ketentuan pemberian Uang\nMakan Dinas\nBiaya Transportasi Dinas SesuaiketentuanpemberianBiayaTransportasiDinas\nBiaya Akomodasi - Sesuai ketentuan pemberian Biaya -\nAkomodasi Dinas\nBiaya Lain-lain Diberikanjikaditetapkanoleh Perusahaan yang ditentukanoleh CEO (untukKaryawanBusiness\nUnit) atauDireksi(untukKaryawanCorporate) sebagaibagiandaribiayaPerjalananDinas.\nCash Advance Tidak Diperbolehkan Diperbolehkan'

In [54]:
len(tables_flattened[3])

555

In [59]:
len(table_6)

554

In [60]:
filtered_flattened_table = [txt for txt in tables_flattened if len(txt)>len(table_6)-10 and len(txt)<len(table_6)+10]
filtered_flattened_table

['Jenis Biaya PDJD PDJD Menginap PDJJ PDJJ Tidak Menginap Uang Saku Dinas - Sesuai ketentuan pemberian Uang Saku Dinas - Uang Makan Dinas - - Sesuai ketentuan pemberian Uang Makan Dinas Biaya Transportasi Dinas SesuaiketentuanpemberianBiayaTransportasiDinas Biaya Akomodasi - Sesuai ketentuan pemberian Biaya Akomodasi Dinas - Biaya Lain-lain Diberikanjikaditetapkanoleh Perusahaan yang ditentukanoleh CEO (untukKaryawanBusiness Unit) atauDireksi(untukKaryawanCorporate) sebagaibagiandaribiayaPerjalananDinas. Cash Advance Tidak Diperbolehkan Diperbolehkan ']

In [66]:
one,score = process.extractOne(table_6, tables_flattened, scorer=fuzz.token_sort_ratio)
score

100

In [67]:
one

'Jenis Biaya PDJD PDJD Menginap PDJJ PDJJ Tidak Menginap Uang Saku Dinas - Sesuai ketentuan pemberian Uang Saku Dinas - Uang Makan Dinas - - Sesuai ketentuan pemberian Uang Makan Dinas Biaya Transportasi Dinas SesuaiketentuanpemberianBiayaTransportasiDinas Biaya Akomodasi - Sesuai ketentuan pemberian Biaya Akomodasi Dinas - Biaya Lain-lain Diberikanjikaditetapkanoleh Perusahaan yang ditentukanoleh CEO (untukKaryawanBusiness Unit) atauDireksi(untukKaryawanCorporate) sebagaibagiandaribiayaPerjalananDinas. Cash Advance Tidak Diperbolehkan Diperbolehkan '

In [69]:
idx = tables_flattened.index(one)
idx

3

In [70]:
tables[idx]

[['Jenis Biaya', 'PDJD', 'PDJD Menginap', 'PDJJ', 'PDJJ Tidak\nMenginap'],
 ['Uang Saku Dinas',
  '-',
  'Sesuai ketentuan pemberian Uang Saku\nDinas',
  None,
  '-'],
 ['Uang Makan Dinas',
  '-',
  '-',
  'Sesuai ketentuan pemberian Uang\nMakan Dinas',
  None],
 ['Biaya Transportasi Dinas',
  'SesuaiketentuanpemberianBiayaTransportasiDinas',
  None,
  None,
  None],
 ['Biaya Akomodasi',
  '-',
  'Sesuai ketentuan pemberian Biaya\nAkomodasi Dinas',
  None,
  '-'],
 ['Biaya Lain-lain',
  'Diberikanjikaditetapkanoleh Perusahaan yang ditentukanoleh CEO (untukKaryawanBusiness\nUnit) atauDireksi(untukKaryawanCorporate) sebagaibagiandaribiayaPerjalananDinas.',
  None,
  None,
  None],
 ['Cash Advance', 'Tidak Diperbolehkan', None, 'Diperbolehkan', None]]

In [46]:
for page_idx, page in enumerate(pdf_pages):
    if page['tables']:
        for i, dct in enumerate(page['tables']):
            if dct['table_dct']:
                table = clean_table(dct['table_dct'])
                print(table)
                headers = table[0]
                data_rows = table[1:]
                # Use a list comprehension to build the list of dictionaries
                records = [dict(zip(headers, row)) for row in data_rows]
            pdf_pages[page_idx]['tables'][i]['table_dct'] = records
#         print(page['page'], ': ', len(page['tables']))

[['', 'PERJALANAN DINAS/ BUSINESS TRAVEL'], ['Dilakukan oleh', 'KaryawanKontrakatau KaryawanTetap.'], ['PersetujuanDinas', '1. PDJDtanpaunsurmenginap:SPPDwajibdisetujuiAtasanLangsungKaryawandenganGolonganminimal6.\n2. PDJD menginap1) : SPPD wajib disetujui Atasan Karyawan minimal Department Head denganmelampirkan justifikasi\ntertulis.\n3. PDJJdalamnegeri:SPPDwajibdisetujuiAtasanKaryawanminimalDepartmentHead.\n4. PDJJ luar negeri: SPPD wajib disetujui hingga CEO untuk Karyawan Business Unit atau Direksi untuk Karyawan\nCorporate.\n1)PDJD dengan unsur menginap dapat dilakukan dengan pertimbangan situasi yang tidak dapat terhindarkan atau\ndipandanglebihefektifdanefisiendarisisibiayadantenaga.'], ['Tugasyang\ndikategorikan\nPerjalananDinas', 'TugaskedinasanyangtermasukdalamkategoriPerjalananDinasantaralain:\n1. MelakukantugaskedinasankegrupusahaPerusahaan.\n2. Mewakili Perusahaan untuk memenuhi undangan dinas instansi pemerintah organisasi/lembaga swasta untuk\nforumsimposium/diskusipane

In [47]:
pdf_pages[6]['tables'][1]['table_dct']

[{'Jenis Biaya': 'Uang Saku Dinas',
  'PDJD': '-',
  'PDJD Menginap': 'Sesuai ketentuan pemberian Uang Saku\nDinas',
  'PDJJ': 'Sesuai ketentuan pemberian Uang Saku\nDinas',
  'PDJJ Tidak\nMenginap': '-'},
 {'Jenis Biaya': 'Uang Makan Dinas',
  'PDJD': '-',
  'PDJD Menginap': '-',
  'PDJJ': 'Sesuai ketentuan pemberian Uang\nMakan Dinas',
  'PDJJ Tidak\nMenginap': 'Sesuai ketentuan pemberian Uang\nMakan Dinas'},
 {'Jenis Biaya': 'Biaya Transportasi Dinas',
  'PDJD': 'SesuaiketentuanpemberianBiayaTransportasiDinas',
  'PDJD Menginap': 'SesuaiketentuanpemberianBiayaTransportasiDinas',
  'PDJJ': 'SesuaiketentuanpemberianBiayaTransportasiDinas',
  'PDJJ Tidak\nMenginap': 'SesuaiketentuanpemberianBiayaTransportasiDinas'},
 {'Jenis Biaya': 'Biaya Akomodasi',
  'PDJD': '-',
  'PDJD Menginap': 'Sesuai ketentuan pemberian Biaya\nAkomodasi Dinas',
  'PDJJ': 'Sesuai ketentuan pemberian Biaya\nAkomodasi Dinas',
  'PDJJ Tidak\nMenginap': '-'},
 {'Jenis Biaya': 'Biaya Lain-lain',
  'PDJD': 'Diberikan